# Completed K=5 run diagnosis
Read-only inspection of the 20261013T002408-4024eb86 run. The tiny-model tests are diagnostics only. All original run files, learning rate, and Figure 2 loss remain unchanged.


In [ ]:
# Offline diagnostics only: no Llama calls, no GPT-2 training, no checkpoint writes.
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import json, math, statistics, csv, gc, hashlib
from pathlib import Path
import numpy as np
import torch
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer
from datasets import Dataset
ROOT = Path.cwd()
while not (ROOT / "Static_K_GRPO_GSM8K.ipynb").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Open inside AUTODSPy repository")
    ROOT = ROOT.parent
RUN = ROOT / "results/gsm8k/k5/seed42/baseline/20261013T002408-4024eb86"
OUT = ROOT / "docs/diagnostics/k5-20261013"
OUT.mkdir(parents=True, exist_ok=True)
rows = [json.loads(s) for s in (RUN / "training.jsonl").read_text().splitlines()]
nb = json.loads((ROOT / "Static_K_GRPO_GSM8K.ipynb").read_text(encoding="utf-8"))
# Execute function definitions only, never initialization/training/inference cells.
for marker in ["MODULES =", "def action_log_probs(", "def group_advantages("]:
    sources = ["".join(c.get("source", [])) for c in nb["cells"]]
    exec(next(s for s in sources if s.startswith(marker)))


In [ ]:
assert [r["episode"] for r in rows] == list(range(1, 31))
data = []
for r in rows:
    candidates = r["candidate_details"]
    assert len(candidates) == 5
    assert [c["trajectory"]["actions"] for c in candidates] == r["actions"]
    assert [c["reward"]["reward"] for c in candidates] == r["rewards"]
    mean, advantages = group_advantages(r["rewards"])
    assert np.allclose(advantages, r["advantages"], atol=1e-14)
    assert math.isclose(statistics.pvariance(r["rewards"]), r["reward_variance"], abs_tol=1e-14)
    recomputed = figure2_loss([c["trajectory"]["log_probs"] for c in candidates], advantages)
    assert math.isclose(recomputed, r["grpo_loss"], abs_tol=1e-6)
    unique = len({tuple(a) for a in r["actions"]})
    data.append(dict(episode=r["episode"], rewards=r["rewards"], unique_pipelines=unique,
        zero_reward_variance=len(set(r["rewards"])) == 1, identical_action_sequences=unique == 1,
        loss_scientific=f"{r['grpo_loss']:.9e}", gradient_norm_scientific=f"{r['gradient_norm']:.9e}"))
keys = list(data[0])
with (OUT / "episodes.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=keys); writer.writeheader(); writer.writerows(data)
for r in data: print(r)
print("PASS: all 30 saved groups pass reward, trajectory, variance, advantage and loss consistency checks")


In [ ]:
# Test the actual notebook replay/update with a tiny deterministic categorical model.
# This tests gradient plumbing; it does not train GPT-2 or execute candidate pipelines.
from types import SimpleNamespace
class TinyPolicy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = torch.nn.Parameter(torch.linspace(-.5, .5, len(ACTIONS)))
        self.config = SimpleNamespace(n_positions=1024)
    def forward(self, **kwargs):
        return SimpleNamespace(logits=self.theta.reshape(1, 1, -1))
class TinyTokenizer:
    def __call__(self, *args, **kwargs):
        return {"input_ids": torch.zeros((1, 1), dtype=torch.long)}
first = {a:i for i,a in enumerate(ACTIONS)}
def session():
    model = TinyPolicy()
    return dict(policy=model, optimizer=torch.optim.Adam(model.parameters(), lr=1e-4, foreach=False),
        tokenizer=TinyTokenizer(), first_tokens=first, config=dict(k=5, max_policy_tokens=1024))
def trajectories(s, actions):
    result = []
    for a in actions:
        t = dict(actions=a)
        t["log_probs"] = [float(x.detach()) for x in replay_log_probs(s["policy"], s["tokenizer"], first, "q", t, s["config"])]
        result.append(t)
    return result
checks = {}
s = session(); ts = trajectories(s, rows[0]["actions"])
advantages = group_advantages(rows[0]["rewards"])[1]
reference = figure2_loss([replay_log_probs(s["policy"], s["tokenizer"], first, "q", t, s["config"]) for t in ts], advantages)
reference.backward(); reference_grad = s["policy"].theta.grad.clone()
expected_norm = float(reference_grad.double().norm())
result = update_policy(s, "q", ts, rows[0]["rewards"])
assert math.isclose(result["gradient_norm"], expected_norm, rel_tol=2e-6)
assert math.isclose(result["loss"], float(reference.detach()), abs_tol=1e-6)
assert result["gradient_norm"] > 0
checks["distinct_trajectories_gradient_norm"] = result["gradient_norm"]
checks["sequential_update_matches_retained_graph"] = True
# Equal rewards have no current gradient, but Adam's existing momentum moves weights.
ts = trajectories(s, [rows[1]["actions"][0]] * 5)
before = s["policy"].theta.detach().clone()
result = update_policy(s, "q", ts, [1.] * 5)
movement = float((s["policy"].theta.detach() - before).double().norm())
assert result["gradient_norm"] == 0 and movement > 0
checks["equal_reward_gradient_norm"] = result["gradient_norm"]
checks["zero_gradient_adam_parameter_movement"] = movement
for e in [11, 15, 16, 27]:
    s = session(); ts = trajectories(s, rows[e-1]["actions"])
    result = update_policy(s, "q", ts, rows[e-1]["rewards"])
    assert abs(result["loss"]) < 1e-6 and result["gradient_norm"] < 1e-6
    assert not result["zero_advantage_group"]
    checks[f"episode_{e}_identical_trajectory_gradient_norm"] = result["gradient_norm"]
(OUT / "gradient-checks.json").write_text(json.dumps(checks, indent=2))
print("PASS", checks)


## Optional offline checkpoint probability inspection
CPU forward passes only; no model download, Llama inference, optimizer updates, or training. Requires the locally cached GSM8K Arrow file and GPT-2 tokenizer/config.


In [ ]:
# Read cached GPT-2 files and checkpoints on CPU. No optimizer step is executed here.
cfg = json.loads((RUN / "config.json").read_text())["config"]
tokenizer = GPT2Tokenizer.from_pretrained(cfg["policy_model"], revision=cfg["policy_revision"], local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token; tokenizer.truncation_side = "left"
first = {a:tokenizer.encode(a, add_special_tokens=False)[0] for a in ACTIONS}
model_cfg = GPT2Config.from_pretrained(cfg["policy_model"], revision=cfg["policy_revision"], local_files_only=True)
model_cfg.pad_token_id = tokenizer.pad_token_id
with torch.device("meta"): model = GPT2LMHeadModel(model_cfg)
arrow = Path.home() / ".cache/huggingface/datasets/openai___gsm8k/main/0.0.0" / cfg["dataset_revision"] / "gsm8k-train.arrow"
train = Dataset.from_file(str(arrow))
inspections = []
optimizer_inspections = []
for file in sorted((RUN / "checkpoints").glob("*.pt")):
    payload = torch.load(file, map_location="cpu", mmap=True, weights_only=False)
    model.load_state_dict(payload["policy"], assign=True); model.eval()
    ep = payload["state"]["episode"]
    if ep:
        states = payload["optimizer"]["state"]
        optimizer_inspections.append(dict(episode=ep,
            step_values=sorted({float(s["step"]) for s in states.values()}),
            first_moment_norm=math.sqrt(sum(float(s["exp_avg"].double().square().sum()) for s in states.values()))))
    for e in [1, 2]:
        row = rows[e-1]; prompt = train[int(row["question_id"].split(":")[-1])]["question"]
        for partial in [[], ["Predict"]]:
            with torch.no_grad(): choices, scores = action_log_probs(model, tokenizer, first, prompt, partial, cfg)
            probs = scores.double().exp()
            inspections.append(dict(checkpoint_episode=ep, prompt_episode=e, partial=partial,
                probabilities=dict(zip(choices, probs.tolist())), entropy_nats=float(-(probs*scores.double()).sum())))
    del payload; gc.collect()
# The exact episode-1 checkpoint is absent. Selected probabilities after one update
# come from episode-2 rollout logs. Module entropy is recoverable (two choices);
# 15-choice signature entropy has rigorous bounds, not an exact observed value.
p = [math.exp(x) for x in rows[1]["candidate_details"][0]["trajectory"]["log_probs"]]
def binary_entropy(x): return -x*math.log(x)-(1-x)*math.log1p(-x)
print("After one update, selected module/signature probabilities:", p[:2])
print("Module entropy (nats):", binary_entropy(p[0]))
print("Signature entropy bounds (nats):", binary_entropy(p[1]), binary_entropy(p[1])+(1-p[1])*math.log(14))
(OUT / "checkpoint-reinspection.json").write_text(json.dumps(dict(checkpoint_evaluations=inspections, optimizer=optimizer_inspections), indent=2))
print("PASS: offline checkpoint inspection complete")


In [ ]:
a = torch.load(next((RUN / "checkpoints").glob("episode_0025*")), map_location="cpu", mmap=True, weights_only=False)
b = torch.load(next((RUN / "checkpoints").glob("episode_0030*")), map_location="cpu", mmap=True, weights_only=False)
seen = set(); changed = 0; squared = 0.; maximum = 0.
for name, x in a["policy"].items():
    if x.data_ptr() in seen: continue
    seen.add(x.data_ptr()); delta = (b["policy"][name] - x).double()
    changed += int(torch.count_nonzero(delta)); squared += float(delta.square().sum())
    maximum = max(maximum, float(delta.abs().max()))
print("25-to-30 weight movement:", changed, math.sqrt(squared), maximum)
del a, b; gc.collect()
